# ラベルと特徴量エンジニアリング

このラボは*オプション（任意）*です。時系列エンジニアリングのための高度な SQL クエリを示します。現実世界の問題において、この種の特徴量エンジニアリングコードは不可欠です。オープンプロジェクトウィークで時系列プロジェクトに取り組む場合は、このコードをテンプレートとして自由に使用してください。 

---

学習目標:

1. BigQuery を使用して予測のための時系列特徴量とラベルを構築する方法を学ぶ
2. 特徴量を可視化および探索する方法を学ぶ
3. モデリング結果を向上させるための効果的なスケーリングと正規化の手法を学ぶ

データを探索したので、モデルを構築できるように特徴量の構築を始めましょう。

<h3><font color="#4885ed">特徴量エンジニアリング</font> </h3>

`price_history` テーブルを使用して、特定の銘柄の過去のパフォーマンスを確認し、将来の株価の予測を試みます。このノートブックでは、このテーブルからのデータのクリーニングと特徴量の作成に焦点を当てます。 

時系列データを使用して特徴量を作成するには、一般的に 2 つの異なるアプローチがあります。 

**1 つ目のアプローチ**は、時系列データを「min_price_over_past_month」や「exp_moving_avg_past_30_days」などの「静的な」特徴量に集約することです。このアプローチを使用すると、ディープニューラルネットワークやより「伝統的な」ML モデルを使用してトレーニングできます。集約した後は、基本的にはすべての時系列順序情報が削除されていることに注目してください。この仮定は実際にはうまく機能することがよくあります。 

**2 つ目のアプローチ**は、データの順序付けられた性質を保持し、リカレントニューラルネットワークなどのシーケンシャルモデルを使用することです。このアプローチには、通常特徴量エンジニアリングの手間が少なくて済むという利点があります。ただし、シーケンシャルモデルのトレーニングには通常より長い時間がかかります。

このノートブックでは、特徴量を構築し、順序付けられた時系列データのローリングウィンドウも作成します。

<h3><font color="#4885ed">ラベルエンジニアリング</font> </h3>

株価が上昇するか下落するかを予測しようとしています。これを行うには、将来のデータを確認し、それをラベルとして使用することで、ラベルを「エンジニアリング（設計）」する必要があります。これを行うために、BigQuery の [`LAG`](https://cloud.google.com/bigquery/docs/reference/standard-sql/functions-and-operators#lag) 関数を使用します。視覚的には次のようになります。

![](https://github.com/GoogleCloudPlatform/training-data-analyst/blob/master/blogs/gcp_forecasting/rolling_window.gif?raw=true)

## ライブラリのインポートとセットアップ

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]

In [ ]:
import pandas as pd
from google.cloud import bigquery
from IPython import get_ipython
from IPython.core.magic import register_cell_magic

%load_ext google.cloud.bigquery

bq = bigquery.Client(project=PROJECT)

In [ ]:
# SQL クエリで Python 変数を簡単に使用できるようにします。
@register_cell_magic("with_globals")
def with_globals(line, cell):
    contents = cell.format(**globals())
    if "print" in line:
        print(contents)
    get_ipython().run_cell(contents)

In [ ]:
def create_dataset():
    dataset = bigquery.Dataset(bq.dataset("stock_market"))
    try:
        bq.create_dataset(dataset)  # データセットがすでに存在する場合は失敗します。
        print("Dataset created")
    except:
        print("Dataset already exists")


create_dataset()

## 時系列特徴量の作成と市場動向に基づくラベルの決定

### ベーステーブルのサマリー

**TODO**: ベーステーブル `price_history` と `snp500` には何行ありますか？

In [ ]:
%%with_globals
%%bigquery --project {PROJECT}
--# TODO

In [ ]:
%%with_globals
%%bigquery --project {PROJECT}
--# TODO

### ラベルエンジニアリング

最終的に、各日に対して 1 つのラベルが必要になります。ラベルは {`down`, `stay`, `up`} の 3 つの値を取ります。ここで `down` と `up` は、それぞれ正規化された価格（詳細については後述）が 1% 以上下落、または 1% 以上上昇したことを示します。`stay` は株価が 1% 以内にとどまったことを示します。

手順は以下のとおりです。

1. 終値（close）と始値（open）を比較する
2. 分析関数を使用して価格特徴量を計算する
3. 正規化された価格変動率（%）を計算する
4. S&P 500 テーブルと結合する
5. ラベル（`up`, `down`, `stay`）を作成する

<h3><font color="#4885ed">終値と始値の比較</font> </h3>

各行について、[`LAG`](https://cloud.google.com/bigquery/docs/reference/legacy-sql#lag) 関数を使用して昨日の終値と明日の始値を取得します。ここでは「明日の終値 - 今日の終値」を求めます。

#### シフトして明日の終値を取得する。

**学習目標 1**

In [ ]:
%%with_globals print
%%bigquery df --project {PROJECT}
CREATE OR REPLACE TABLE `stock_market.price_history_delta`
AS
(
WITH shifted_price AS
  (
  SELECT *,
  (LAG(close, 1) OVER (PARTITION BY symbol order by Date DESC)) AS tomorrow_close
  FROM `stock_src.price_history`
  WHERE Close > 0
  )
SELECT a.*,
(tomorrow_close - Close) AS tomo_close_m_close
FROM shifted_price a
)

In [ ]:
%%with_globals
%%bigquery --project {PROJECT}
SELECT *
FROM stock_market.price_history_delta
ORDER by Date
LIMIT 100

**TODO**: 歴史的に株式市場が上昇傾向にあることは知られています。新しく作成したテーブル `price_history_delta` を使用してこれを確認する方法を考えられますか？

**学習目標 2**

In [ ]:
%%with_globals print
%%bigquery --project {PROJECT}
SELECT
--# TODO: 株式市場が平均して上昇していることを確認します。
FROM
stock_market.price_history_delta

### 時系列特徴量の追加

<h3><font color="#4885ed">分析関数を使用した価格特徴量の計算</font> </h3>

さらに、最小値（min）、最大値（max）、平均値（mean）、標準偏差（std）を使用して時系列特徴量も構築します（他に使える OVER 関数は思いつきますか？）。これを行うために、BigQuery の[分析関数（analytic functions）]()（ウィンドウ関数とも呼ばれます）を使用しましょう。 
```
分析関数は、行のグループに対して集計値を計算する関数です。行のグループに対して単一の集計値を返す集約関数とは異なり、分析関数は入力行のグループに対して関数を計算することにより、各行に対して単一の値を返します。
```
 `AVG` 分析関数を使用すると、特定のシンボルについて過去 1 週間（5 営業日）の平均終値を計算できます。
```python
 (AVG(close) OVER (PARTITION BY symbol
  ORDER BY Date
  ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING)) / close
  AS close_avg_prior_5_days
```

**学習目標 1**

**TODO**: 以下のクエリの `# TODO` を入力してください。

In [ ]:
def get_window_fxn(agg_fxn, n_days):
    """Generate a time-series feature.

    E.g., Compute the average of the price over the past 5 days."""
    SCALE_VALUE = "close"
    sql = """
    ({agg_fxn}(close) OVER (PARTITION BY (# TODO)
                      ORDER BY (# TODO)
                      ROWS BETWEEN {n_days} (# TODO)))/{scale}
                      AS close_{agg_fxn}_prior_{n_days}_days""".format(
        agg_fxn=agg_fxn, n_days=n_days, scale=SCALE_VALUE
    )
    return sql


WEEK = 5
MONTH = 20
YEAR = 52 * 5

agg_funcs = ("MIN", "MAX", "AVG", "STDDEV")
lookbacks = (WEEK, MONTH, YEAR)
sqls = []

for fxn in agg_funcs:
    for lookback in lookbacks:
        sqls.append(get_window_fxn(fxn, lookback))
time_series_features_sql = ",".join(sqls)  # SQL 文字列。


def preview_query():
    print(time_series_features_sql[0:1000])


preview_query()

In [ ]:
%%with_globals print
%%bigquery --project {PROJECT}
CREATE OR REPLACE TABLE stock_market.price_features_delta
AS
SELECT * 
FROM 
    (SELECT *,
    {time_series_features_sql},
    -- Also get the raw time-series values; will be useful for the RNN model.
    (ARRAY_AGG(close) OVER (PARTITION BY symbol
                    ORDER BY Date
                    ROWS BETWEEN 260 PRECEDING AND 1 PRECEDING))
                    AS close_values_prior_260,
    ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY Date) AS days_on_market
    FROM stock_market.price_history_delta)
WHERE days_on_market > {YEAR}

In [ ]:
%%bigquery --project {PROJECT}
SELECT *
FROM stock_market.price_features_delta 
ORDER BY symbol, Date
LIMIT 10

#### 変動率（%）を計算し、S&P 指数の価格と自己結合する。 

S&P 指数である GSPC の価格変動も計算します。これを行うことで、正規化された変動率（%）を計算できます。

<h3><font color="#4885ed">正規化された価格変動率（%）の計算</font> </h3>

ラベルを作成する前に、S&P 500 指数を使用して価格変動を正規化する必要があります。S&P インデックスファンドを使用した正規化は、個別の銘柄の将来の価格変動が市場全体の影響によるものではないことを確認するのに役立ちます。正規化により、株式市場のパフォーマンスに寄与する要因を分離できます。

 同じ期間における S&P 500 指数のスケーリングされた差分を差し引く正規化スキームを使用しましょう。
 
 Python で表すと次のようになります: 
```python
# 計算例。
scaled_change = (50.59 - 50.69) / 50.69
scaled_s_p = (939.38 - 930.09) / 930.09
normalized_change = scaled_change - scaled_s_p
assert normalized_change == ~1.2%
```

In [ ]:
scaled_change = (50.59 - 50.69) / 50.69
scaled_s_p = (939.38 - 930.09) / 930.09
normalized_change = scaled_change - scaled_s_p
print(
    """
scaled change: {:2.3f}
scaled_s_p: {:2.3f}
normalized_change: {:2.3f}
""".format(
        scaled_change, scaled_s_p, normalized_change
    )
)

### 正規化された価格変動の計算（上記参照）。

スケーリングされた価格変動（tomorrow_close / close）を [gspc](https://en.wikipedia.org/wiki/S%26P_500_Index) シンボル（S&P 指数のシンボル）と結合しましょう。その後、上記で説明したスキームを使用して正規化できます。

**学習目標 3**

**TODO**: 以下のコードの `# TODO` を入力してください。

In [ ]:
snp500_index = "gspc"

In [ ]:
%%with_globals print
%%bigquery --project {PROJECT}
CREATE OR REPLACE TABLE stock_market.price_features_norm_per_change
AS
WITH 
all_percent_changes AS
    (
    SELECT *, (tomo_close_m_close / Close) AS scaled_change
    FROM `stock_market.price_features_delta`
    ),
s_p_changes AS
    (SELECT
     scaled_change AS s_p_scaled_change,
     date
     FROM all_percent_changes
     WHERE symbol="{snp500_index}")

SELECT all_percent_changes.*,
       s_p_scaled_change,
       (
           # TODO
       ) AS normalized_change          
FROM 
    all_percent_changes LEFT JOIN s_p_changes
    --# すべての行に S&P の変動を追加
    ON all_percent_changes.date = s_p_changes.date

#### 結果の確認

In [ ]:
%%with_globals print
%%bigquery df --project {PROJECT}
SELECT *
FROM stock_market.price_features_norm_per_change
LIMIT 10

In [ ]:
df.head()

<h3><font color="#4885ed">S&P 500 テーブルとの結合とラベルの作成: {`up`, `down`, `stay`}</font> </h3>

テーブルを S&P 500 のリストと結合します。これにより、分析対象を S&P 500 構成企業のみに限定できます。

最後にラベルを作成します。以下の SQL 文を使用します:

```sql
  CASE WHEN normalized_change < -0.01 THEN 'DOWN'
       WHEN normalized_change > 0.01 THEN 'UP'
       ELSE 'STAY'
  END
 ```

**学習目標 1**

In [ ]:
down_thresh = -0.01
up_thresh = 0.01

**TODO**: 以下の `CASE` 文を入力してください。

In [ ]:
%%with_globals print
%%bigquery df --project {PROJECT}
CREATE OR REPLACE TABLE stock_market.percent_change_sp500
AS
SELECT *,
    CASE 
    --# TODO
    END AS direction
FROM stock_market.price_features_norm_per_change features
INNER JOIN `stock_src.snp500`
USING (symbol)

In [ ]:
%%with_globals print
%%bigquery --project {PROJECT}
SELECT direction, COUNT(*) as cnt
FROM stock_market.percent_change_sp500
GROUP BY direction

In [ ]:
%%with_globals print
%%bigquery df --project {PROJECT}
SELECT *
FROM stock_market.percent_change_sp500
LIMIT 20

In [ ]:
df.columns

データセットは依然として非常に大きく、市場の大部分の日は `STAY` となります。企業によって[一株当たり利益（EPS）](https://en.wikipedia.org/wiki/Earnings_per_share) 情報が発表された日付に分析を集中させましょう。EPS データには、surprise、reported_EPS、consensus_EPS という 3 つの主要な列があります。

In [ ]:
%%with_globals print
%%bigquery --project {PROJECT}
SELECT *
FROM `stock_src.eps`
LIMIT 10

surprise 列は、予想値（アナリストによるコンセンサス予想 EPS）と報告された EPS の差を示します。このテーブルを派生テーブルと結合して、決算発表期間中の分析に集中させることができます。

In [ ]:
%%with_globals print
%%bigquery --project {PROJECT}
CREATE OR REPLACE TABLE stock_market.eps_percent_change_sp500
AS
SELECT a.*, b.consensus_EPS, b.reported_EPS, b.surprise
FROM stock_market.percent_change_sp500 a
INNER JOIN `stock_src.eps` b
ON a.Date = b.date
AND a.symbol = b.symbol

In [ ]:
%%with_globals print
%%bigquery --project {PROJECT}
SELECT *
FROM stock_market.eps_percent_change_sp500
LIMIT 20

In [ ]:
%%with_globals print
%%bigquery --project {PROJECT}
SELECT direction, COUNT(*) as cnt
FROM stock_market.eps_percent_change_sp500
GROUP BY direction

## 特徴量の探索

企業の株価の直近の動きを作成したので、特徴量を可視化してみましょう。これにより、データをより深く理解でき、計算中に行った可能性のあるエラーを発見できる場合があります。

念のため、予測対象の日付の 1 週間前、1 か月前、1 年前のスケーリングされた価格を計算したことを思い出してください。

特徴量を集計するための再利用可能な関数を作成しましょう。

**学習目標 2**

In [ ]:
def get_aggregate_stats(field, round_digit=2):
    """Run SELECT ... GROUP BY field, rounding to nearest digit."""
    df = bq.query(
        """
    SELECT {field}, COUNT(*) as cnt
    FROM
    (SELECT ROUND({field}, {round_digit}) AS {field}
    FROM stock_market.eps_percent_change_sp500) rounded_field
    GROUP BY {field}
    ORDER BY {field}""".format(
            field=field, round_digit=round_digit, PROJECT=PROJECT
        )
    ).to_dataframe()
    return df.dropna()

In [ ]:
field = "close_AVG_prior_260_days"
CLIP_MIN, CLIP_MAX = 0.1, 4.0
df = get_aggregate_stats(field)
values = df[field].clip(CLIP_MIN, CLIP_MAX)
counts = 100 * df["cnt"] / df["cnt"].sum()  # パーセンテージ。
ax = values.hist(weights=counts, bins=30, figsize=(10, 5))
ax.set(xlabel=field, ylabel="%");

**TODO**: 上記の `get_aggregate_stats` を使用して `normalized_change` 列を可視化してください。

In [ ]:
field = "normalized_change"
# TODO

曜日や月などごとに結果を確認してみましょう。

In [ ]:
VALID_GROUPBY_KEYS = (
    "DAYOFWEEK",
    "DAY",
    "DAYOFYEAR",
    "WEEK",
    "MONTH",
    "QUARTER",
    "YEAR",
)
DOW_MAPPING = {
    1: "Sun",
    2: "Mon",
    3: "Tues",
    4: "Wed",
    5: "Thur",
    6: "Fri",
    7: "Sun",
}


def groupby_datetime(groupby_key, field):
    if groupby_key not in VALID_GROUPBY_KEYS:
        raise Exception("Please use a valid groupby_key.")
    sql = """
    SELECT {groupby_key}, AVG({field}) as avg_{field}
    FROM
    (SELECT {field},
     EXTRACT({groupby_key} FROM date) AS {groupby_key}
    FROM stock_market.eps_percent_change_sp500) foo
    GROUP BY {groupby_key}
    ORDER BY {groupby_key} DESC""".format(
        groupby_key=groupby_key, field=field, PROJECT=PROJECT
    )
    print(sql)
    df = bq.query(sql).to_dataframe()
    if groupby_key == "DAYOFWEEK":
        df.DAYOFWEEK = df.DAYOFWEEK.map(DOW_MAPPING)
    return df.set_index(groupby_key).dropna()

In [ ]:
field = "normalized_change"

df = groupby_datetime("DAYOFWEEK", field)
ax = df.plot(kind="barh", color="orange", alpha=0.7)
ax.grid(which="major", axis="y", linewidth=0)

In [ ]:
field = "close"
df = groupby_datetime("DAYOFWEEK", field)
ax = df.plot(kind="barh", color="orange", alpha=0.7)
ax.grid(which="major", axis="y", linewidth=0)

In [ ]:
field = "normalized_change"
df = groupby_datetime("MONTH", field)
ax = df.plot(kind="barh", color="blue", alpha=0.7)
ax.grid(which="major", axis="y", linewidth=0)

In [ ]:
field = "normalized_change"
df = groupby_datetime("QUARTER", field)
ax = df.plot(kind="barh", color="green", alpha=0.7)
ax.grid(which="major", axis="y", linewidth=0)

In [ ]:
field = "close"
df = groupby_datetime("YEAR", field)
ax = df.plot(kind="line", color="purple", alpha=0.7)
ax.grid(which="major", axis="y", linewidth=0)

In [ ]:
field = "normalized_change"
df = groupby_datetime("YEAR", field)
ax = df.plot(kind="line", color="purple", alpha=0.7)
ax.grid(which="major", axis="y", linewidth=0)

ボーナス: 特徴量はラベル `direction` とどのように相関していますか？いくつかの可視化を作成してみてください。最も重要な特徴量は何でしょうか？[`CORR`](https://cloud.google.com/bigquery/docs/reference/standard-sql/statistical_aggregate_functions) 関数を使用してこれを可視化し、統計的に評価できます。

Copyright 2019 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License